# Holiday Feature Engineering

## Objective

Prepare a documented Brazilian holiday calendar and create
order-level holiday features for purchase-time prediction.

## Inputs

- data/processed/research_data.csv
- Historical holiday calendars

## Outputs

- data/external/processed/brazil_national_holidays.csv
- data/processed/order_holiday_features.csv

## Rules

- Keep one row per order_id.
- Distinguish national holidays from optional holidays.
- Do not use actual delivery dates to create predictors.
- Preserve missing values when calendar coverage is insufficient.

In [1]:
# Purpose: Locate the project and load dates needed for holiday features.

from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

current_dir = Path.cwd()

if (current_dir / "data/processed/research_data.csv").exists():
    PROJECT_ROOT = current_dir
elif (current_dir.parent / "data/processed/research_data.csv").exists():
    PROJECT_ROOT = current_dir.parent
else:
    raise FileNotFoundError("Cannot locate research_data.csv.")

DATA_DIR = PROJECT_ROOT / "data/processed"
EXTERNAL_DIR = PROJECT_ROOT / "data/external/processed"

EXTERNAL_DIR.mkdir(parents=True, exist_ok=True)

orders = pd.read_csv(
    DATA_DIR / "research_data.csv",
    usecols=[
        "order_id",
        "order_purchase_timestamp",
        "order_estimated_delivery_date",
    ],
    dtype={"order_id": "string"},
)

assert orders["order_id"].notna().all()
assert orders["order_id"].str.strip().ne("").all()
assert orders["order_id"].is_unique

date_columns = [
    "order_purchase_timestamp",
    "order_estimated_delivery_date",
]

for column in date_columns:
    orders[column] = pd.to_datetime(
        orders[column],
        errors="raise",
    )

display(orders.head())

print("Total orders:", len(orders))
print("Earliest purchase:", orders["order_purchase_timestamp"].min())
print("Latest purchase:", orders["order_purchase_timestamp"].max())
print(
    "Latest estimated delivery:",
    orders["order_estimated_delivery_date"].max(),
)

,order_id,order_purchase_timestamp,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,2017-10-02 10:56:33,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,2018-07-24 20:41:37,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,2018-08-08 08:38:49,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,2017-11-18 19:28:06,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,2018-02-13 21:18:39,2018-02-26


Total orders: 99441
Earliest purchase: 2016-09-04 21:15:19
Latest purchase: 2018-10-17 17:30:18
Latest estimated delivery: 2018-11-12 00:00:00


In [2]:
# Purpose: Build the documented holiday calendar for 2016–2018.

holiday_sources = {
    2016: (
        "https://progep.ufc.br/pt/"
        "mpog-divulga-calendario-de-feriados-nacionais-"
        "e-pontos-facultativos-de-2016/"
    ),
    2017: (
        "https://progep.ufc.br/pt/"
        "portaria-define-calendario-dos-feriados-nacionais-"
        "e-pontos-facultativos-de-2017/"
    ),
    2018: (
        "https://www.ifpb.edu.br/campus/cajazeiras/ensino/"
        "calendario-academico/documentos/"
        "calendario-integrados-2018-cajazeiras-campus.pdf/"
        "@@download/file"
    ),
}

fixed_holidays = {
    "01-01": "Confraternizacao Universal",
    "04-21": "Tiradentes",
    "05-01": "Dia Mundial do Trabalho",
    "09-07": "Independencia do Brasil",
    "10-12": "Nossa Senhora Aparecida",
    "11-02": "Finados",
    "11-15": "Proclamacao da Republica",
    "12-25": "Natal",
}

good_friday_dates = {
    2016: "2016-03-25",
    2017: "2017-04-14",
    2018: "2018-03-30",
}

holiday_rows = []

for year, source_url in holiday_sources.items():
    annual_holidays = [
        (f"{year}-{month_day}", name)
        for month_day, name in fixed_holidays.items()
    ]

    annual_holidays.append(
        (good_friday_dates[year], "Paixao de Cristo")
    )

    for date, name in annual_holidays:
        holiday_rows.append({
            "holiday_date": date,
            "holiday_name": name,
            "holiday_type": "national_in_federal_calendar",
            "source_year": year,
            "source_url": source_url,
        })

holiday_calendar = pd.DataFrame(holiday_rows)

holiday_calendar["holiday_date"] = pd.to_datetime(
    holiday_calendar["holiday_date"],
    format="%Y-%m-%d",
    errors="raise",
)

holiday_calendar = (
    holiday_calendar
    .sort_values("holiday_date")
    .reset_index(drop=True)
)

assert holiday_calendar["holiday_date"].notna().all()
assert holiday_calendar["holiday_date"].is_unique
assert holiday_calendar["source_url"].notna().all()

display(holiday_calendar[
    ["holiday_date", "holiday_name", "source_year"]
])

display(
    holiday_calendar.groupby("source_year")
    .size()
    .rename("holiday_count")
    .to_frame()
)

,holiday_date,holiday_name,source_year
0,2016-01-01,Confraternizacao Universal,2016
1,2016-03-25,Paixao de Cristo,2016
2,2016-04-21,Tiradentes,2016
3,2016-05-01,Dia Mundial do Trabalho,2016
4,2016-09-07,Independencia do Brasil,2016
5,2016-10-12,Nossa Senhora Aparecida,2016
6,2016-11-02,Finados,2016
7,2016-11-15,Proclamacao da Republica,2016
8,2016-12-25,Natal,2016
9,2017-01-01,Confraternizacao Universal,2017


,holiday_count
source_year,
2016,9
2017,9
2018,9


In [5]:
# Purpose: Create order-level holiday features using calendar dates.

purchase_date = orders["order_purchase_timestamp"].dt.normalize()
estimated_date = orders["order_estimated_delivery_date"].dt.normalize()

calendar_start = pd.Timestamp("2016-01-01")
calendar_end = pd.Timestamp("2018-12-31")

holiday_days = (
    holiday_calendar["holiday_date"]
    .sort_values()
    .to_numpy(dtype="datetime64[ns]")
)

purchase_days = purchase_date.to_numpy(dtype="datetime64[ns]")
estimated_days = estimated_date.to_numpy(dtype="datetime64[ns]")

order_holidays = orders[["order_id"]].copy()

# Feature 1: Purchase occurs on a holiday.
purchase_covered = purchase_date.between(
    calendar_start, calendar_end
)

order_holidays["purchase_is_national_holiday"] = (
    purchase_date.isin(holiday_calendar["holiday_date"])
    .astype("Int64")
    .where(purchase_covered)
)

# Feature 2: Days until the next holiday, including the purchase date.
next_positions = np.searchsorted(
    holiday_days, purchase_days, side="left"
)

next_holiday_date = pd.Series(
    pd.NaT, index=orders.index, dtype="datetime64[ns]"
)

has_next_holiday = (
    purchase_covered & (next_positions < len(holiday_days))
)

next_holiday_date.loc[has_next_holiday] = holiday_days[
    next_positions[has_next_holiday.to_numpy()]
]

order_holidays["days_to_next_national_holiday"] = (
    (next_holiday_date - purchase_date)
    .dt.days
    .astype("Int64")
)

# Feature 3: Holidays after purchase and up to the estimated date.
valid_window = (
    purchase_covered
    & estimated_date.between(calendar_start, calendar_end)
    & estimated_date.ge(purchase_date)
)

holidays_after_purchase = np.searchsorted(
    holiday_days, purchase_days, side="right"
)

holidays_through_estimate = np.searchsorted(
    holiday_days, estimated_days, side="right"
)

order_holidays["national_holiday_count_in_estimated_window"] = (
    pd.Series(
        holidays_through_estimate - holidays_after_purchase,
        index=orders.index,
        dtype="Int64",
    )
    .where(valid_window)
)

display(order_holidays.head(10))

,order_id,purchase_is_national_holiday,days_to_next_national_holiday,national_holiday_count_in_estimated_window
0,e481f51cbdc54678b7cc49136f2d6af7,0,10,1
1,53cdb2fc8bc7dce0b6741e2150273451,0,45,0
2,47770eb9100c2d0c44946d9cf07ec65d,0,30,0
3,949d5b44dbf5de918fe9c16f97b45f8a,0,37,0
4,ad21c59c0840e6cb83a9ceb5573f8159,0,45,0
5,a4591c265e18cb1dcee52889e2d8acc3,0,60,0
6,136cce7faa42fdb2cefd53fdc79a6098,0,3,3
7,6514b8ad8028c9f2cc2374ded245783f,0,114,0
8,76c6e866289321a7c93b82b54852dc33,0,81,0
9,e69bfb5eb88e0ed6a785585b27e16dbf,0,40,0


In [6]:
# Purpose: Validate holiday features and inspect missing values.

holiday_features = [
    "purchase_is_national_holiday",
    "days_to_next_national_holiday",
    "national_holiday_count_in_estimated_window",
]

assert len(order_holidays) == len(orders)
assert order_holidays["order_id"].is_unique
assert order_holidays["order_id"].equals(orders["order_id"])

assert (
    order_holidays["purchase_is_national_holiday"]
    .dropna().isin([0, 1]).all()
)

for column in holiday_features[1:]:
    assert order_holidays[column].dropna().ge(0).all()

# Independently verify the holiday count using direct date comparisons.
expected_counts = pd.Series(
    pd.NA, index=orders.index, dtype="Int64"
)

expected_counts.loc[valid_window] = 0

for holiday_date in holiday_calendar["holiday_date"]:
    included = (
        valid_window
        & purchase_date.lt(holiday_date)
        & estimated_date.ge(holiday_date)
    )
    expected_counts.loc[included] += 1

pd.testing.assert_series_equal(
    order_holidays["national_holiday_count_in_estimated_window"],
    expected_counts,
    check_names=False,
)

display(pd.DataFrame({
    "missing_rows": order_holidays[holiday_features].isna().sum(),
    "missing_percent": (
        order_holidays[holiday_features].isna().mean() * 100
    ).round(2),
}))

display(
    order_holidays["purchase_is_national_holiday"]
    .value_counts(dropna=False)
    .rename_axis("holiday_flag")
    .to_frame("order_count")
)

print("All holiday checks passed.")

,missing_rows,missing_percent
purchase_is_national_holiday,0,0.0
days_to_next_national_holiday,0,0.0
national_holiday_count_in_estimated_window,0,0.0


,order_count
holiday_flag,
0,97909
1,1532


All holiday checks passed.


In [8]:
# Purpose: Save both holiday datasets in data/processed.

calendar_path = DATA_DIR / "brazil_national_holidays.csv"
features_path = DATA_DIR / "order_holiday_features.csv"

holiday_calendar.to_csv(
    calendar_path,
    index=False,
    date_format="%Y-%m-%d",
)

order_holidays.to_csv(features_path, index=False)

print("Calendar saved:", calendar_path.resolve())
print("Calendar exists:", calendar_path.exists())
print("Calendar rows:", len(holiday_calendar))

print("Order features saved:", features_path.resolve())

Calendar saved: C:\Users\LENOVO\Downloads\Group5_STAT3013.R11-main (1)\Group5_STAT3013.R11-main\data\processed\brazil_national_holidays.csv
Calendar exists: True
Calendar rows: 27
Order features saved: C:\Users\LENOVO\Downloads\Group5_STAT3013.R11-main (1)\Group5_STAT3013.R11-main\data\processed\order_holiday_features.csv
